In [11]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException

In [ ]:
def checkFileExistorNot(savepath):
    first_date = None
    old_html_rows = ""

    if os.path.exists(savepath):
        old_df = pd.read_html(savepath, attrs={"id": "myTableCPriceHistory"})[0]
        first_date = pd.to_datetime(old_df.iloc[0, 1])

        with open(savepath, "r", encoding="utf-8") as f:
            old_content = f.read()

        old_html_rows = old_content.split("<tbody>")[1].split("</tbody>")[0]

    return first_date, old_html_rows

stock_ignored = [ "CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO"]

filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
df = pd.read_csv(filepath)
symbols = df['Symbol'].tolist()

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service("/usr/bin/chromedriver"), options=options)
driver.set_page_load_timeout(30)

print(f"Found {len(symbols)} companies to scrape")

for symbol in symbols:

    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    print(f"\nScraping {symbol}...")

    savepath = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/NEPSE-StockWebScrap/{symbol}.html"
    first_date, old_html_rows = checkFileExistorNot(savepath)

    try:
        driver.get(f"https://www.sharesansar.com/company/{symbol}")
        time.sleep(4)

        driver.find_element(By.LINK_TEXT, "Price History").click()
        time.sleep(4)

        all_new_rows = []
        page = 1
        stop_scraping = False

        while True:
            rows = driver.find_elements(By.CSS_SELECTOR, "#myTableCPriceHistory tbody tr")

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                scraped_date = pd.to_datetime(cols[1].text)
                
                if first_date is None:
                    all_new_rows.append(row.get_attribute("outerHTML"))
                else:
                    if scraped_date > first_date:
                        print(f"scrapping date: {scraped_date}  ")
                        all_new_rows.append(row.get_attribute("outerHTML"))
                    else:
                        stop_scraping = True
                        break

            if stop_scraping:
                break

            next_button = driver.find_element(By.ID, "myTableCPriceHistory_next")
            if "disabled" in next_button.get_attribute("class"):
                break

            next_button.click()
            print(f"✓ page: {page} ")
            time.sleep(3)
            page += 1

        if first_date is None:
            final_rows = ''.join(all_new_rows)
        else:
            final_rows = ''.join(all_new_rows) + old_html_rows

        if final_rows.strip():
            html_content = f"""<table id="myTableCPriceHistory">
<tbody>
{final_rows}
</tbody>
</table>"""

            os.makedirs(os.path.dirname(savepath), exist_ok=True)

            with open(savepath, "w", encoding="utf-8") as f:
                f.write(html_content)

            print(f"Saved {len(all_new_rows)} new rows")
        else:
            print("No new data found")

    except TimeoutException:
        print("Page load timeout")
    except Exception as e:
        print(f"Error scraping {symbol}: {e}")

driver.quit()
print("All done!")